<a href="https://colab.research.google.com/github/Hardeep-Randhawa/Generative-AI-Practices/blob/main/NewPrograms/audio-video-translation/audio_to_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Approach 1: The Cascaded Pipeline (Highly Recommended for Low Memory)

In [1]:
# 1. Install prerequisites
!pip install transformers torch torchaudio soundfile datasets --quiet

In [2]:

import torch
import torchaudio
import soundfile as sf
from transformers import pipeline, VitsModel, AutoTokenizer
from huggingface_hub import login


In [ ]:


def lightweight_speech_to_speech(audio_input_path: str, output_path: str = "translated_output.wav"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running on device: {device}")

    # Step 1: Automatic Speech Recognition (English Audio -> English Text)
    print("⏳ Loading Whisper-Tiny for transcription...")
    asr_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-tiny", device=device)

    print("Transcribing audio...")
    transcription = asr_pipe(audio_input_path)["text"]
    print(f"📝 Original Text: {transcription}")

    # Step 2: Translation (Using 'text2text-generation' to fix the KeyError)
    print("⏳ Loading Helsinki-NLP Translation Engine...")
    translation_pipe = pipeline("text2text-generation", model="Helsinki-NLP/opus-mt-en-es", device=device)

    print("Translating text...")
    translated_text = translation_pipe(transcription)[0]["generated_text"] # 'generated_text' matches the new task format
    print(f"🔮 Translated Text: {translated_text}")

    # Step 3: Text-to-Speech (Spanish Text -> Spanish Audio)
    print("⏳ Loading Meta MMS Text-to-Speech...")
    tts_model = VitsModel.from_pretrained("facebook/mms-tts-spa").to(device)
    tts_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-spa")

    print("Synthesizing final target voice waveform...")
    inputs = tts_tokenizer(text=translated_text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_audio = tts_model(**inputs).waveform.squeeze().cpu().numpy()

    # Step 4: Write the file out
    sf.write(output_path, output_audio, tts_model.config.sampling_rate)
    print(f"✨ Success! Translated audio file saved as: {output_path}")

# --- Execution ---
# Run this directly with your uploaded MP3 file
lightweight_speech_to_speech("/content/english_speech.mp3", output_path="translated_output.wav")

In [5]:
import torchaudio
import os

# Path to the audio file
audio_file_path = "/content/english_speech.mp3"

# Check if the file exists
if not os.path.exists(audio_file_path):
    print(f"Error: The file '{audio_file_path}' does not exist.")
else:
    try:
        # Load the audio file and get its metadata
        waveform, sample_rate = torchaudio.load(audio_file_path)

        print(f"Successfully loaded audio file: {audio_file_path}")
        print(f"Sample rate: {sample_rate} Hz")
        print(f"Number of channels: {waveform.shape[0]}")
        print(f"Number of frames: {waveform.shape[1]}")
        print(f"Duration: {waveform.shape[1] / sample_rate:.2f} seconds")

        # You can also play a snippet to confirm if it's audible
        from IPython.display import Audio
        print("Playing first 5 seconds of the audio:")
        display(Audio(waveform[:, :sample_rate * 5], rate=sample_rate))

    except Exception as e:
        print(f"Error loading audio file with torchaudio: {e}")
        print("This might indicate a corrupted file or an unsupported codec.")

Successfully loaded audio file: /content/english_speech.mp3
Sample rate: 24000 Hz
Number of channels: 1
Number of frames: 254016
Duration: 10.58 seconds
Playing first 5 seconds of the audio:


If the above cell reports an error loading the audio or playing it, the `.mp3` file itself might be the issue. If it loads successfully, then the problem might lie in how the `transformers` pipeline processes it, possibly due to a version mismatch or an internal dependency.